# `run_embedders` — every argument, and what it changes

One function turns reference datasets into gene embeddings:

```python
from panelprep.embed import run_embedders

run_embedders(datasets,              # {name: CandidateDataset}, each G x N_k
              *, embedder_names,     # which methods
              genes,                 # the gene universe, |G| of them
              dim=32,                # d
              params=None,           # per-embedder overrides
              seed=0,
              max_obs=None)          # -> Dict[str, GeneEmbedding]
```

Each returned `GeneEmbedding.matrix` is `|G| x d` — **one row per gene, indexed by gene symbol**.

This notebook changes **one argument at a time** and shows what moves in the output: which keys come
back, what shape they are, how many rows are NaN, and — where the geometry itself changes — *how
much* it changed, measured with CKA and kNN-Jaccard rather than eyeballed.

Everything below runs on the synthetic fixture, which is small, fast and has a known answer by
construction, so the effect of each argument is not confounded with real-data noise. The last two
sections repeat the key comparisons on the real pilot embeddings loaded from the store.

In [ ]:
import os, sys, inspect, warnings
from pathlib import Path
if not Path('panelprep').is_dir():
    os.chdir('..')
sys.path.append(os.getcwd())

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
warnings.filterwarnings('ignore', category=FutureWarning)

from panelprep.embed import (EMBEDDERS, run_embedders, gene_by_cell, gene_universe,
                            describe, compare)
from panelprep.embed.store import EmbeddingStore
from panelprep.io.synthetic import make_synthetic_case
import panelprep.integrations.morpholab as morpholab   # registers factor_analysis

print(inspect.signature(run_embedders))

## The inputs

`datasets` is a dict of `CandidateDataset`. Each holds an **observations x genes** matrix — rows are
cells, columns are gene symbols. `run_embedders` transposes internally to the `G x N_k` gene-by-cell
form, because in the embedding half of this package the **gene** is the observation and the cells
are the features.

In [ ]:
case = make_synthetic_case(seed=0)
datasets = case.candidates
pd.DataFrame([{'dataset': k, 'category': v.category, 'n_obs': v.n_obs,
               'n_genes': len(v.genes), 'has_coords': v.has_coords, 'units': v.units}
              for k, v in datasets.items()])

In [ ]:
# The gene universe: what every returned matrix will be indexed by.
genes = gene_universe(datasets, how='intersection', max_genes=150,
                      rng=np.random.default_rng(0), must_include=list(case.query.panel))
print(f'{len(genes)} genes; panel {list(case.query.panel)} all present: '
      f'{all(g in genes for g in case.query.panel)}')

# What one embedder actually consumes, for one dataset:
gxn, mask = gene_by_cell(datasets['sibling'], genes)
print(f'gene_by_cell -> {gxn.shape} (G x N), presence mask {mask.sum()}/{len(mask)} genes present')

## 1. `embedder_names` — which keys come back

The registry records a `mode` per embedder, and that single field decides the shape of the result:

- `mode='separate'` → the embedder is called **once per dataset**, and you get one matrix per
  dataset, keyed `<method>::<dataset>`.
- `mode='joint'` → the embedder is called **once with all datasets**, and you get **one** matrix,
  keyed `<method>`, built from every dataset as a view.

So asking for three separate methods over four datasets returns twelve matrices; asking for three
joint methods returns three.

In [ ]:
EMBEDDERS.meta_table()[['mode', 'category', 'needs_expression']]

In [ ]:
SEP = ['pca_baseline', 'factor_analysis']        # mode='separate'
JOINT = ['joint_ae', 'dcca']                     # mode='joint'

out_sep = run_embedders(datasets, embedder_names=SEP, genes=genes, dim=8, seed=0)
out_joint = run_embedders(datasets, embedder_names=JOINT, genes=genes, dim=8, seed=0)

print(f'{len(SEP)} separate methods x {len(datasets)} datasets -> {len(out_sep)} matrices')
print(f'{len(JOINT)} joint methods                 -> {len(out_joint)} matrices')
describe({**out_sep, **out_joint})[['key', 'method', 'mode', 'trained_on', 'shape', 'n_nan_genes']]

### What changes if you add one name

Adding an embedder name adds keys and touches nothing else — the existing matrices are bit-identical,
because each embedder is seeded independently from `(seed, method name)`.

In [ ]:
out_more = run_embedders(datasets, embedder_names=SEP + ['vae_separate'],
                         genes=genes, dim=8, seed=0)
added = sorted(set(out_more) - set(out_sep))
print(f'keys added: {len(added)}')
for k in added: print('   ', k)

same = all(np.allclose(out_more[k].matrix.to_numpy(), out_sep[k].matrix.to_numpy(),
                       equal_nan=True) for k in out_sep)
print(f'\npre-existing matrices unchanged: {same}')

## 2. `genes` — the index of every returned matrix

The index of every matrix is **exactly the list you passed, in that order**. This is what lets you
compare two embeddings row-by-row without aligning anything.

A gene absent from a dataset is not dropped and is not zero-filled — it comes back as an **all-NaN
row**. A zero vector is a position in the embedding space, and zero-filling would silently place
every absent gene at the same point, in the middle of the data.

In [ ]:
subset = genes[:40]
out_small = run_embedders(datasets, embedder_names=['pca_baseline'], genes=subset, dim=8, seed=0)
emb = out_small['pca_baseline::sibling']
print(f'asked for {len(subset)} genes -> matrix {emb.matrix.shape}')
print(f'index identical to what was passed: {list(emb.matrix.index) == list(subset)}')

In [ ]:
# Ask for genes that do not exist in the data at all.
fake = ['NotAGene1', 'NotAGene2']
out_fake = run_embedders(datasets, embedder_names=['pca_baseline', 'factor_analysis'],
                         genes=subset + fake, dim=8, seed=0)
d = describe(out_fake)
print(d[['key', 'n_genes', 'n_nan_genes']].to_string(index=False))
print('\nthe fabricated genes, in factor_analysis:')
display(out_fake['factor_analysis::sibling'].matrix.loc[fake].iloc[:, :5])

Both embedders return the fabricated genes as all-NaN rows — `n_nan_genes` is 2 for every matrix above — so an absent gene is never silently placed somewhere in the space. They reach that by different routes, which the provenance records: `factor_analysis` sets the rows NaN explicitly after fitting, while `pca_baseline` tracks the presence mask. Read `n_nan_genes` from `describe()` rather than assuming either behaviour.

In [ ]:
for k in ['pca_baseline::sibling', 'factor_analysis::sibling']:
    p = out_fake[k].provenance
    keep = {kk: p[kk] for kk in ('n_genes_present', 'n_genes_absent_set_nan',
                                 'n_obs', 'dim_used', 'status') if kk in p}
    print(f'{k}: {keep}')

## 3. `dim` — the width `d`, and how much survives the squeeze

`dim` sets the number of columns. The question worth asking is not what shape comes back — that is
obvious — but **how much of the gene geometry is preserved** as `d` falls. Comparing each setting
against the widest one answers it.

`compare()` is rotation- and column-order-invariant, so a high value means the two embeddings really
place genes in the same arrangement rather than merely having the same shape.

In [ ]:
rows = []
ref = run_embedders(datasets, embedder_names=['pca_baseline'], genes=genes,
                    dim=32, seed=0)['pca_baseline::sibling']
for d_ in [2, 4, 8, 16, 32]:
    e = run_embedders(datasets, embedder_names=['pca_baseline'], genes=genes,
                      dim=d_, seed=0)['pca_baseline::sibling']
    rows.append({'dim': d_, 'shape': f'{e.matrix.shape[0]} x {e.matrix.shape[1]}',
                 'cka_vs_d32': round(compare(e, ref, metric='cka'), 4),
                 'knn_jaccard_vs_d32': round(compare(e, ref, metric='knn_jaccard', k=10), 4)})
dim_sweep = pd.DataFrame(rows)
dim_sweep

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4), constrained_layout=True)
ax.plot(dim_sweep['dim'], dim_sweep['cka_vs_d32'], 'o-', label='CKA (global geometry)')
ax.plot(dim_sweep['dim'], dim_sweep['knn_jaccard_vs_d32'], 's-',
        label='kNN-Jaccard, k=10 (local neighbourhoods)')
ax.set_xscale('log', base=2); ax.set_xlabel('dim (d)')
ax.set_ylabel('agreement with d=32'); ax.set_ylim(0, 1.02)
ax.set_title('What `dim` costs you', fontsize=11); ax.legend(fontsize=8, frameon=False)
fig

The gap between the two curves is the point: CKA recovers quickly because the dominant axes of
variation are captured in a few components, while kNN-Jaccard lags — the *local* neighbourhoods,
which is what a downstream nearest-neighbour method actually consumes, keep changing after the
global shape has settled. Pick `dim` against the metric your downstream step uses.

## 4. `datasets` — and why it changes joint embedders differently

For a `separate` embedder, removing a dataset removes its matrix and leaves the rest untouched.

For a `joint` embedder there is only one matrix, built from **all** the datasets at once, so
removing one changes the single returned embedding. That is the whole difference between the two
modes, and it is easy to forget when reading a figure.

In [ ]:
subset_ds = {k: v for k, v in datasets.items() if k != 'unrelated'}

sep_all = run_embedders(datasets, embedder_names=['pca_baseline'], genes=genes, dim=8, seed=0)
sep_few = run_embedders(subset_ds, embedder_names=['pca_baseline'], genes=genes, dim=8, seed=0)
print(f'separate: {len(sep_all)} matrices -> {len(sep_few)} after dropping one dataset')
shared = set(sep_all) & set(sep_few)
print('surviving matrices identical:',
      all(np.allclose(sep_all[k].matrix.to_numpy(), sep_few[k].matrix.to_numpy(),
                      equal_nan=True) for k in shared))

joint_all = run_embedders(datasets, embedder_names=['dcca'], genes=genes, dim=8, seed=0)['dcca']
joint_few = run_embedders(subset_ds, embedder_names=['dcca'], genes=genes, dim=8, seed=0)['dcca']
print(f'\njoint: 1 matrix either way, but the matrix itself moves')
print(f'   CKA(all views, fewer views)          = {compare(joint_all, joint_few, metric="cka"):.4f}')
print(f'   kNN-Jaccard(all views, fewer views)  = '
      f'{compare(joint_all, joint_few, metric="knn_jaccard", k=10):.4f}')
print(f'   views recorded in provenance: {joint_all.provenance.get("views")}')

## 5. `params` — per-embedder overrides

`params` is `{embedder_name: {kwarg: value}}`, merged over the registry defaults. Only the named
embedder is affected. The defaults themselves are in `EMBEDDERS[name].meta['defaults']`, so you can
always see what you are overriding.

In [ ]:
for name in ['pca_baseline', 'factor_analysis', 'contrastive_ae']:
    print(f"{name:18s} {EMBEDDERS[name].meta['defaults']}")

In [ ]:
base = run_embedders(datasets, embedder_names=['pca_baseline'], genes=genes, dim=8, seed=0)
tweaked = run_embedders(datasets, embedder_names=['pca_baseline'], genes=genes, dim=8, seed=0,
                        params={'pca_baseline': {'standardize': False}})
k = 'pca_baseline::sibling'
print(f"standardize=True vs False on the same data:")
print(f"   CKA         = {compare(base[k], tweaked[k], metric='cka'):.4f}")
print(f"   kNN-Jaccard = {compare(base[k], tweaked[k], metric='knn_jaccard', k=10):.4f}")
print('\nper-column z-scoring is not cosmetic: without it the embedding is dominated by '
      'whichever cells have the largest total counts.')

## 6. `seed` and `max_obs`

`seed` matters only for the stochastic embedders — the neural ones and any dataset subsampling.
A deterministic factorisation returns the same answer whatever the seed, and that asymmetry is worth
knowing before you report a seed-to-seed difference as a finding.

`max_obs` caps the number of observations (cells) used per dataset. Fewer cells means a noisier
estimate of the gene-gene relationships, so the embedding moves.

In [ ]:
rows = []
for name in ['pca_baseline', 'joint_ae']:
    a = run_embedders(datasets, embedder_names=[name], genes=genes, dim=8, seed=0)
    b = run_embedders(datasets, embedder_names=[name], genes=genes, dim=8, seed=7)
    key = name if name in a else f'{name}::sibling'
    rows.append({'embedder': name, 'mode': EMBEDDERS[name].meta['mode'],
                 'cka_seed0_vs_seed7': round(compare(a[key], b[key], metric='cka'), 4),
                 'knn_seed0_vs_seed7': round(compare(a[key], b[key], metric='knn_jaccard', k=10), 4)})
pd.DataFrame(rows)

In [ ]:
rows = []
full = run_embedders(datasets, embedder_names=['pca_baseline'], genes=genes, dim=8, seed=0)
for mo in [100, 250, 500, None]:
    e = run_embedders(datasets, embedder_names=['pca_baseline'], genes=genes, dim=8,
                      seed=0, max_obs=mo)['pca_baseline::sibling']
    rows.append({'max_obs': mo if mo else 'all',
                 'n_obs_used': e.provenance.get('n_obs'),
                 'cka_vs_full': round(compare(e, full['pca_baseline::sibling'], metric='cka'), 4),
                 'knn_vs_full': round(compare(e, full['pca_baseline::sibling'],
                                              metric='knn_jaccard', k=10), 4)})
pd.DataFrame(rows)

## 7. Reading the provenance

Every embedding carries a provenance dict. It is the record of what was actually done — not what was
requested — so it is where you look when a number surprises you.

In [ ]:
e = run_embedders(datasets, embedder_names=['dcca'], genes=genes, dim=8, seed=0)['dcca']
for k, v in e.provenance.items():
    sv = str(v)
    print(f'  {k:32s} {sv[:95]}{"..." if len(sv) > 95 else ""}')

## 8. The same function on the real pilot, loaded from the store

The real embeddings were produced by exactly this call — `run_embedders` over seven mouse embryo
reference datasets, 2000 genes, `dim=32`. They are read back here instead of recomputed; the
`EmbeddingStore` round-trips the matrix, the category and the provenance.

In [ ]:
store = EmbeddingStore('outputs/real_pilot/store', strict=False)
print(store)
real = store.load_all(group='expression')
real = {k: v for k, v in real.items() if v is not None}
describe(real)[['key', 'method', 'mode', 'trained_on', 'category', 'shape', 'n_nan_genes']]

In [ ]:
# Same method, different dataset vs different method, same genes.
pairs = [
    ('pca_baseline::seu3d_E7.5_rep2', 'pca_baseline::seu3d_E7.75_rep2', 'same method, sibling embryo'),
    ('pca_baseline::seu3d_E7.5_rep2', 'pca_baseline::mittnenzweig_wt_E7.5', 'same method, other platform'),
    ('pca_baseline::seu3d_E7.5_rep2', 'dcca', 'separate vs integrated'),
    ('dcca', 'joint_ae', 'two joint methods'),
]
rows = []
for a, b, label in pairs:
    if a in real and b in real:
        rows.append({'comparison': label, 'a': a, 'b': b,
                     'cka': round(compare(real[a], real[b], metric='cka'), 4),
                     'knn_jaccard': round(compare(real[a], real[b], metric='knn_jaccard', k=15), 4)})
pd.DataFrame(rows)

Two things to take from that table. CKA stays high where kNN-Jaccard collapses — the global
geometry can agree while the local neighbourhoods, the thing most downstream methods consume, do
not. And the cross-platform row is far lower than the cross-embryo row: **the assay, not the
developmental stage, is the dominant axis** in these embeddings.

There is no null model in `compare()`. Use `panelprep.viz.embedding_agreement_matrix`, which
computes the same metrics against a gene-label-shuffled null, before calling any of these values
high or low.

## 9. Adding your own embedder

Register a function and it appears in `run_embedders` immediately — no edit anywhere else. A
`separate` embedder is called once per dataset with a single-entry `datasets` mapping and returns
**one** `GeneEmbedding`; `run_embedders` supplies the `<method>::<dataset>` key.

In [ ]:
from panelprep.schemas import GeneEmbedding

@EMBEDDERS.register(
    'row_mean_demo', category='single_cell', mode='separate', needs_expression=True,
    defaults=dict(), description='toy embedder: per-gene summary statistics, for the docs')
def row_mean_demo(datasets, *, genes, dim, rng, max_obs=None, **_):
    """Mean, sd and detection rate per gene, padded to `dim` columns."""
    name, ds = next(iter({k: v for k, v in datasets.items() if v.has_expression}.items()))
    gxn, mask = gene_by_cell(ds, genes, max_obs=max_obs, rng=rng)
    X = gxn.to_numpy(float)
    feats = np.column_stack([X.mean(1), X.std(1), (X > 0).mean(1)])
    M = np.full((len(genes), dim), np.nan, dtype='float32')
    M[:, :min(dim, 3)] = feats[:, :min(dim, 3)]
    M[~mask, :] = np.nan
    return GeneEmbedding(matrix=pd.DataFrame(M, index=list(genes)),
                         method='row_mean_demo', category=ds.category,
                         provenance={'dataset': name, 'status': 'implemented',
                                     'n_genes_present': int(mask.sum())})

out_new = run_embedders(datasets, embedder_names=['row_mean_demo'], genes=genes, dim=8, seed=0)
describe(out_new)[['key', 'method', 'mode', 'trained_on', 'shape', 'n_nan_genes', 'status']]

In the package this goes in `panelprep/embed/custom.py`, and the name goes into
`Config.embedders` — after which it flows through `run_analysis`, the figure gallery and the
comparison report with no further edits. Put it in `panelprep.run.CACHED_ONLY` if it is expensive
and depends only on the gene list.

## Summary — what each argument changes

| argument | changes | does **not** change |
|---|---|---|
| `embedder_names` | which keys come back; `mode` decides one-per-dataset vs one-total | the matrices of the other embedders |
| `genes` | the index of every matrix, exactly and in order; absent genes become NaN rows | which datasets are used |
| `dim` | the number of columns; how much local structure survives | the index |
| `datasets` | how many matrices for `separate`; **the matrix itself** for `joint` | the gene index |
| `params` | only the named embedder's behaviour | every other embedder |
| `seed` | stochastic embedders and subsampling only | deterministic factorisations |
| `max_obs` | how many cells inform the estimate, so the geometry | shapes and keys |